<a href="https://colab.research.google.com/github/devilistiani/coding_camp_copilot/blob/naila/genai_feature_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generative AI Feature — Intent-Aware Auto Reply

Notebook ini menambahkan fitur **Generative AI** sebagai *fitur sekunder* pada aplikasi,
sesuai kriteria Side Quest: *"Menggunakan API Generative AI untuk fitur tambahan atau fitur sekunder pada aplikasi"*.

**Alur kerja:**
1. User mengirim pertanyaan
2. Model BiLSTM (dari notebook utama) mengklasifikasi *intent*
3. Intent + pertanyaan dikirim ke **Gemini API** untuk menghasilkan jawaban otomatis yang kontekstual


## 1. Install & Import Dependencies

In [15]:
# Install google-generativeai jika belum tersedia
!pip install -q google-generativeai


In [16]:
import os
import re
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
import google.generativeai as genai

print("Dependencies loaded successfully.")


Dependencies loaded successfully.


## 2. Load Model dari Notebook Utama

Pastikan `copilot_intent_model_best.keras` tersedia di environment yang sama.
Jika di Google Colab, upload file `.keras` terlebih dahulu.


In [17]:
class AttentionPooling(tf.keras.layers.Layer):
    """
    Custom Layer: Attention-weighted pooling.
    Memberikan bobot perhatian pada tiap token sehingga kata penting
    mendapat kontribusi lebih besar ke representasi akhir.
    """
    def __init__(self, **kwargs):
        super(AttentionPooling, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name='attention_weight',
            shape=(input_shape[-1], 1),
            initializer='glorot_uniform',
            trainable=True
        )
        self.b = self.add_weight(
            name='attention_bias',
            shape=(1,),
            initializer='zeros',
            trainable=True
        )
        super(AttentionPooling, self).build(input_shape)

    def call(self, inputs):
        score   = tf.nn.tanh(tf.matmul(inputs, self.W) + self.b)
        weights = tf.nn.softmax(score, axis=1)
        context = tf.reduce_sum(weights * inputs, axis=1)
        return context

    def get_config(self):
        return super(AttentionPooling, self).get_config()


class FocalLoss(tf.keras.losses.Loss):
    """
    Custom Loss: Focal Loss untuk menangani class imbalance.
    FL(p_t) = -alpha * (1 - p_t)^gamma * log(p_t)
    """
    def __init__(self, gamma=2.0, alpha=0.25, name='focal_loss', **kwargs):
        super(FocalLoss, self).__init__(name=name, **kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_true    = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred    = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        num_cls   = tf.shape(y_pred)[-1]
        y_true_oh = tf.one_hot(y_true, num_cls)
        p_t       = tf.reduce_sum(y_true_oh * y_pred, axis=-1)
        focal_w   = self.alpha * tf.pow(1.0 - p_t, self.gamma)
        return tf.reduce_mean(focal_w * (-tf.math.log(p_t)))

    def get_config(self):
        cfg = super(FocalLoss, self).get_config()
        cfg.update({'gamma': self.gamma, 'alpha': self.alpha})
        return cfg


# Load model
MODEL_PATH = 'copilot_intent_model_best.keras'

intent_model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={'AttentionPooling': AttentionPooling, 'FocalLoss': FocalLoss}
)
print(f"Model loaded from: {MODEL_PATH}")
intent_model.summary()


Model loaded from: copilot_intent_model_best.keras


Model: "IntentClassifier_v1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input_Pertanyaan (InputLayer)   │ (None, 80)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Embedding (Embedding)           │ (None, 80, 64)         │       192,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ BiLSTM (Bidirectional)          │ (None, 80, 128)        │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ AttentionPooling                │ (None, 128)            │           129 │
│ (AttentionPooling)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Intent (Dense)           │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 268,645 (1.02 MB)

 Trainable params: 268,645 (1.02 MB)

 Non-trainable params: 0 (0.00 B)

## 3. Preprocessing Pipeline (identik dengan notebook utama)

In [18]:
# Hyperparameter (harus sama persis dengan notebook utama)
VOCAB_SIZE    = 3000
MAX_LENGTH    = 80
TRUNCTYPE     = 'post'
PADDING_TYPE  = 'post'
OOV_TOK       = '<OOV>'
EMBEDDING_DIM = 64

LABEL_NAMES = {
    0: 'Administrasi & Akun',
    1: 'Capstone & Reporting',
    2: 'Materi & Kurikulum',
    3: 'Teknis/Lain-lain'
}

# Slang dict (disinkronkan penuh dengan notebook utama)
SLANG_DICT = {
    # Sapaan & filler
    'kak': '', 'kakak': '', 'min': '', 'bang': '', 'gan': '', 'bro': '',
    'halo': '', 'hai': '', 'hei': '', 'permisi': '', 'selamat': '', 'malam': '', 'pagi': '',
    'siang': '', 'sore': '', 'wkwk': '', 'wkwkwk': '', 'hehe': '', 'haha': '', 'hihi': '',
    'hehehe': '', 'dong': '', 'deh': '', 'sih': '', 'loh': '', 'lho': '', 'neh': '', 'si': '',
    # Kata ganti informal
    'aku': 'saya', 'gue': 'saya', 'gw': 'saya', 'sy': 'saya',
    'lo': 'anda', 'lu': 'anda', 'kamu': 'anda', 'km': 'kamu', 'kalian': 'anda', 'kita': 'kami',
    # Negasi & penegasan informal
    'gak': 'tidak', 'ga': 'tidak', 'ngga': 'tidak',
    'nggak': 'tidak', 'gk': 'tidak', 'tdk': 'tidak',
    'tak': 'tidak', 'yaa': 'ya', 'yap': 'ya', 'iyaa': 'iya',
    'oke': 'ok', 'oks': 'ok', 'okey': 'ok',
    # Kata kerja & ekspresi informal
    'nanya': 'bertanya', 'nanyain': 'menanyakan', 'gimana': 'bagaimana', 'gmn': 'bagaimana',
    'udah': 'sudah', 'udh': 'sudah', 'sdh': 'sudah', 'uda': 'sudah',
    'blm': 'belum', 'belom': 'belum', 'blum': 'belum',
    'dapet': 'dapat', 'dpt': 'dapat', 'tuh': 'itu', 'tu': 'itu', 'nih': 'ini', 'ni': 'ini',
    'kayak': 'seperti', 'kaya': 'seperti',
    'kalo': 'kalau', 'klo': 'kalau', 'krn': 'karena', 'karna': 'karena', 'krna': 'karena',
    'emg': 'memang', 'emang': 'memang', 'aja': 'saja', 'aj': 'saja', 'doang': 'saja',
    'tp': 'tapi', 'tpi': 'tapi', 'ttg': 'tentang',
    'yg': 'yang', 'yng': 'yang', 'lg': 'lagi', 'lgi': 'lagi',
    'jg': 'juga', 'jga': 'juga', 'dgn': 'dengan', 'dg': 'dengan', 'utk': 'untuk',
    'bgt': 'banget', 'bngt': 'banget', 'banget': 'sangat',
    'bkn': 'bukan', 'dr': 'dari', 'dri': 'dari', 'kpn': 'kapan', 'knp': 'kenapa',
    'kenapa': 'mengapa', 'bikin': 'membuat', 'buat': 'membuat',
    'mau': 'ingin', 'mo': 'ingin', 'pengen': 'ingin', 'pengin': 'ingin',
    'make': 'menggunakan', 'pake': 'menggunakan', 'pakai': 'menggunakan',
    'ngelakuin': 'melakukan', 'ngeliat': 'melihat', 'ngerti': 'mengerti',
    'ngerasa': 'merasa', 'nyoba': 'mencoba', 'coba': 'mencoba',
    'nyari': 'mencari', 'cari': 'mencari',
    'liat': 'lihat', 'lihat': 'melihat',
    'tau': 'tahu', 'tw': 'tahu',
    'kmrn': 'kemarin', 'kemaren': 'kemarin',
    'stlh': 'setelah', 'abis': 'setelah', 'habis': 'setelah',
    'sehabis': 'setelah', 'abisnya': 'setelahnya',
    'ntar': 'nanti', 'tar': 'nanti',
    # Lain-lain
    'kelar': 'selesai', 'beres': 'selesai', 'fix': 'pasti',
    'mantap': 'bagus', 'mantul': 'bagus', 'gacor': 'bagus',
    'bocil': 'pemula', 'gapapa': 'tidak apa', 'gpp': 'tidak apa',
    'amit amit': 'buruk',
}

STOP_WORDS = set([
    'yang', 'dan', 'di', 'ke', 'dari', 'pada', 'untuk', 'dengan', 'ini',
    'itu', 'atau', 'juga', 'ada', 'saya', 'kami', 'anda', 'kita', 'ia',
    'dia', 'mereka', 'adalah', 'akan', 'dapat', 'bisa', 'oleh', 'dalam',
    'sudah', 'telah', 'sedang', 'pun', 'lagi', 'saja', 'jadi', 'kalau',
    'jika', 'apakah', 'agar', 'tentang', 'bahwa', 'maka', 'namun', 'tetapi',
    'tapi', 'seperti', 'tersebut', 'hal', 'cara', 'lebih', 'sangat', 'harus',
    'tidak', 'belum', 'masih', 'ingin', 'boleh', 'bukan', 'hanya', 'setiap',
    'semua', 'banyak', 'bagaimana', 'mengapa', 'kapan', 'siapa', 'berapa',
    'ya', 'ok', 'iya', 'oh', 'nanti', 'karena', 'sehingga',
])


def normalize_slang(text: str) -> str:
    words = text.split()
    normalized, i = [], 0
    while i < len(words):
        if i + 1 < len(words):
            bigram = words[i] + ' ' + words[i + 1]
            if bigram in SLANG_DICT:
                r = SLANG_DICT[bigram]
                if r:
                    normalized.append(r)
                i += 2
                continue
        word = words[i]
        r = SLANG_DICT.get(word, word)
        if r:
            normalized.append(r)
        i += 1
    return ' '.join(normalized)


def remove_stop_words(text: str) -> str:
    tokens = text.split()
    # BUGFIX: separator diperbaiki dari '  ' (double space) menjadi ' ' (single space)
    return ' '.join([w for w in tokens if w not in STOP_WORDS and len(w) > 1])


def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = normalize_slang(text)
    text = remove_stop_words(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


print("Preprocessing pipeline ready.")


Preprocessing pipeline ready.


## 4. Re-fit Tokenizer

> Tokenizer Keras tidak bisa di-serialize langsung bersama model `.keras`,
> sehingga perlu di-fit ulang menggunakan dataset yang sama agar vocabulary konsisten.


In [19]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer

# Google Colab  : '/content/Dataset Capstone Project - Sheet1.csv'
# Lokal / Drive : sesuaikan path-nya
CSV_PATH = '/content/Dataset Capstone Project - Sheet1.csv'

df_raw = pd.read_csv(CSV_PATH)

df_raw = df_raw.drop(columns=[c for c in df_raw.columns if 'Unnamed' in str(c)], errors='ignore')
df_raw = df_raw.dropna(how='all').reset_index(drop=True)

df_tok = df_raw[['Pertanyaan']].dropna().copy()
df_tok = df_tok[df_tok['Pertanyaan'].str.strip() != '']
df_tok['Pertanyaan_Clean'] = df_tok['Pertanyaan'].apply(clean_text)

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOK)
tokenizer.fit_on_texts(df_tok['Pertanyaan_Clean'].tolist())

print(f"Dataset dimuat: {len(df_raw)} baris.")
print(f"Tokenizer re-fitted. Vocab size: {len(tokenizer.word_index):,} kata unik.")


Dataset dimuat: 297 baris.
Tokenizer re-fitted. Vocab size: 1,524 kata unik.


## 5. Konfigurasi Gemini API

In [20]:
import os

# Ambil API key
# Opsi 1 (Colab Secrets):
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    # Opsi 2: environment variable (untuk lokal / VS Code / Jupyter)
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')

# Opsi 3: hardcode sementara

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY belum diset. Tambahkan di Colab Secrets atau environment variable.")

genai.configure(api_key=GEMINI_API_KEY)

GEMINI_MODEL_NAME = 'models/gemini-3.1-flash-lite'

gemini_model = genai.GenerativeModel(
    model_name=GEMINI_MODEL_NAME,
    generation_config=genai.types.GenerationConfig(
        temperature=0.35,         # sedikit lebih deterministic untuk jawaban faktual
        max_output_tokens=512,    # ditingkatkan agar jawaban tidak terpotong
        top_p=0.9,
    ),
    system_instruction=(
        "Kamu adalah asisten program Coding Camp DBS Foundation. "
        "Selalu jawab dalam Bahasa Indonesia yang ramah, jelas, dan ringkas. "
        "Jika tidak mengetahui jawaban pasti, sarankan peserta menghubungi admin program."
    ),
)

print(f"Gemini API configured successfully: {GEMINI_MODEL_NAME}")

# Verifikasi model tersedia
try:
    gemini_model.generate_content("tes")
    print("Koneksi ke Gemini API berhasil.")
except Exception as e:
    print(f"[PERINGATAN] {e}")
    print("Mencoba fallback ke gemini-2.5-flash ...")
    GEMINI_MODEL_NAME = 'models/gemini-2.5-flash'
    gemini_model = genai.GenerativeModel(
        model_name=GEMINI_MODEL_NAME,
        generation_config=genai.types.GenerationConfig(
            temperature=0.35,
            max_output_tokens=512,
            top_p=0.9,
        ),
        system_instruction=(
            "Kamu adalah asisten program Coding Camp DBS Foundation. "
            "Selalu jawab dalam Bahasa Indonesia yang ramah, jelas, dan ringkas. "
            "Jika tidak mengetahui jawaban pasti, sarankan peserta menghubungi admin program."
        ),
    )
    print(f"Fallback berhasil: {GEMINI_MODEL_NAME}")


Gemini API configured successfully: models/gemini-3.1-flash-lite
Koneksi ke Gemini API berhasil.


## 6. Fungsi Inti: Intent Classification + Generative Reply

**Alur pipeline lengkap:**
```
pertanyaan user - clean_text() - BiLSTM + Attention - intent label + confidence - system_prompt per kategori - Gemini API  →  jawaban otomatis kontekstual
```


In [21]:
# System prompt per kategori intent

SYSTEM_PROMPTS = {
    0: (
        "Konteks: Pertanyaan ini berkaitan dengan ADMINISTRASI & AKUN program Coding Camp DBS Foundation. "
        "Topik meliputi: akun Dicoding, email Credentials, password, data diri, "
        "dosen pembimbing, MBKM, surat keterangan, dan administrasi umum. "
        "Berikan jawaban praktis dan langkah-langkah yang jelas jika diperlukan."
    ),
    1: (
        "Konteks: Pertanyaan ini berkaitan dengan CAPSTONE PROJECT & REPORTING program Coding Camp DBS Foundation. "
        "Topik meliputi: pembentukan tim, dataset, laporan teknis, tech stack, "
        "briefing capstone, mekanisme pengumpulan, dan deadline. "
        "Berikan jawaban teknis yang akurat; sebutkan dokumen/link resmi jika relevan."
    ),
    2: (
        "Konteks: Pertanyaan ini berkaitan dengan MATERI & KURIKULUM program Coding Camp DBS Foundation. "
        "Topik meliputi: modul pembelajaran, kelas, submission, sertifikat, "
        "framework, library, learning path, dan jadwal materi. "
        "Berikan jawaban yang akurat sesuai kurikulum program."
    ),
    3: (
        "Konteks: Pertanyaan ini berkaitan dengan TEKNIS / OPERASIONAL program Coding Camp DBS Foundation. "
        "Topik meliputi: ILT (Instructor-Led Training), kehadiran, reschedule, banding, "
        "daily check-in, dashboard, konsultasi mingguan, dan poin reward. "
        "Berikan jawaban operasional yang jelas dan actionable."
    ),
}


def classify_intent(text: str) -> dict:
    """
    Mengklasifikasi intent pertanyaan menggunakan model BiLSTM.

    Parameters
    ----------
    text : str
        Pertanyaan mentah dari user.

    Returns
    -------
    dict
        label, category, confidence, cleaned_text
    """
    cleaned = clean_text(text)

    # BUGFIX: tangani teks kosong setelah cleaning agar tidak crash
    if not cleaned.strip():
        cleaned = text.lower().strip()

    seq   = tokenizer.texts_to_sequences([cleaned])
    pad   = pad_sequences(seq, maxlen=MAX_LENGTH,
                          padding=PADDING_TYPE, truncating=TRUNCTYPE)
    proba = intent_model.predict(pad, verbose=0)
    label = int(np.argmax(proba))

    return {
        'label'       : label,
        'category'    : LABEL_NAMES[label],
        'confidence'  : float(np.max(proba)),
        'cleaned_text': cleaned,
    }


def generate_reply(question: str, intent_result: dict,
                   max_retries: int = 3, retry_delay: float = 5.0) -> str:
    """
    Menghasilkan jawaban otomatis menggunakan Gemini API,
    dengan konteks intent yang sudah diklasifikasi oleh model BiLSTM.

    PERBAIKAN:
    - Prompt distrukturisasi dengan role/context/question/instruction yang terpisah
    - Ditambah instruksi format jawaban yang konsisten
    - Exponential backoff diperbaiki agar tidak melampaui max_retries

    Parameters
    ----------
    question      : str   Pertanyaan asli user (tidak di-clean)
    intent_result : dict  Output dari classify_intent()
    max_retries   : int   Jumlah maksimal percobaan ulang (default: 3)
    retry_delay   : float Jeda awal antar percobaan dalam detik (default: 5.0)

    Returns
    -------
    str  Jawaban yang dihasilkan Gemini
    """
    import time

    label         = intent_result['label']
    context       = SYSTEM_PROMPTS[label]
    category      = intent_result['category']
    confidence    = intent_result['confidence']

    # PERBAIKAN: prompt distrukturisasi lebih baik
    full_prompt = (
        f"{context}\n\n"
        f"---\n"
        f"Kategori intent terdeteksi : {category} (confidence: {confidence*100:.1f}%)\n"
        f"Pertanyaan peserta         : {question}\n"
        f"---\n\n"
        f"Instruksi jawaban:\n"
        f"1. Jawab langsung pertanyaan di atas dalam Bahasa Indonesia.\n"
        f"2. Gunakan poin/langkah jika jawaban melibatkan prosedur.\n"
        f"3. Jika ada dokumen/link resmi yang relevan, sebutkan.\n"
        f"4. Jangan mengulang pertanyaan; langsung berikan jawabannya.\n"
        f"5. Panjang jawaban: 2–5 kalimat atau poin (tidak perlu terlalu panjang).\n"
    )

    for attempt in range(1, max_retries + 1):
        try:
            response = gemini_model.generate_content(full_prompt)

            # BUGFIX: cek response.text ada sebelum strip
            if response and response.text:
                return response.text.strip()
            else:
                return "[Error] Gemini tidak menghasilkan teks jawaban."

        except Exception as e:
            err_str = str(e)
            # PERBAIKAN: tangkap juga error 400 (invalid argument) dan 404 (model not found)
            is_retryable = any(code in err_str for code in ["503", "429", "500", "overloaded"])
            if is_retryable and attempt < max_retries:
                wait = retry_delay * (2 ** (attempt - 1))  # true exponential backoff
                print(f"  [Retry {attempt}/{max_retries}] {err_str[:80]}... "
                      f"Menunggu {wait:.0f}s")
                time.sleep(wait)
            else:
                print(f"  [Gagal setelah {attempt} percobaan] {err_str[:100]}")
                return f"[Error] Tidak dapat menghasilkan jawaban: {err_str[:80]}"


def intent_aware_reply(question: str) -> dict:
    """
    Pipeline lengkap: klasifikasi intent → generate reply.

    Parameters
    ----------
    question : str  Pertanyaan mentah dari user

    Returns
    -------
    dict berisi question, intent_label, category, confidence, cleaned_text, reply
    """
    intent_result = classify_intent(question)
    reply         = generate_reply(question, intent_result)

    return {
        'question'    : question,
        'intent_label': intent_result['label'],
        'category'    : intent_result['category'],
        'confidence'  : intent_result['confidence'],
        'cleaned_text': intent_result['cleaned_text'],   # TAMBAHAN: untuk debugging
        'reply'       : reply,
    }


print("Pipeline intent_aware_reply() siap digunakan.")


Pipeline intent_aware_reply() siap digunakan.


## 7. Demo — Uji Coba Pipeline

In [22]:
# Demo — Uji Coba Pipeline
test_questions = [
    "kak izin nanya, aku hadir di ilt english tapi telat, bisa diaju banding?",
    "halo, dataset untuk capstone boleh pakai data sintetis dari AI?",
    "password saya tidak bisa dipakai untuk login ke platform dicoding",
    "kapan deadline submission materi tensorflow minggu ini?",
    "gimana cara dapetin sertifikat kalau udah selesai semua kelas?",
]

print("=" * 70)
for q in test_questions:
    result = intent_aware_reply(q)
    print(f"\n Pertanyaan   : {result['question']}")
    print(f" Teks bersih  : {result['cleaned_text']}")   # untuk debugging preprocessing
    print(f" Intent       : [{result['intent_label']}] {result['category']} "
          f"(confidence: {result['confidence']*100:.1f}%)")
    print(f" Auto-Reply   :\n{result['reply']}")
    print("-" * 70)



 Pertanyaan   : kak izin nanya, aku hadir di ilt english tapi telat, bisa diaju banding?
 Teks bersih  : izin bertanya hadir ilt english telat diaju banding
 Intent       : [3] Teknis/Lain-lain (confidence: 57.4%)
 Auto-Reply   :
Halo! Terkait kehadiran ILT yang terlambat, kamu bisa mengajukan banding melalui formulir resmi yang tersedia di dashboard peserta. Berikut langkah-langkahnya:

1. Pastikan kamu memiliki bukti pendukung (seperti *screenshot* kendala teknis atau alasan mendesak).
2. Buka menu **"Banding"** atau **"Support"** di dashboard Coding Camp kamu.
3. Isi formulir pengajuan banding dengan detail yang jelas dan lampirkan bukti tersebut.

Mohon tunggu proses verifikasi dari tim admin. Jika kamu mengalami kendala teknis saat mengakses formulir banding, silakan hubungi admin program melalui kanal komunikasi resmi (Discord/Email) untuk bantuan lebih lanjut.
----------------------------------------------------------------------

 Pertanyaan   : halo, dataset untuk capstone bo

## 8. Batch Evaluation — Sampel dari Dataset Asli

Evaluasi pipeline menggunakan 10 sampel acak dari dataset asli
yang sudah memiliki kolom `Jawaban` sebagai referensi.


In [23]:
import random

random.seed(42)

# PERBAIKAN: pastikan df_raw sudah bersih dari baris header duplikat
df_eval_pool = df_raw[['Pertanyaan', 'Jawaban']].copy()
df_eval_pool = df_eval_pool[
    df_eval_pool['Pertanyaan'].notna() &
    df_eval_pool['Jawaban'].notna() &
    (df_eval_pool['Pertanyaan'].str.strip() != '') &
    (df_eval_pool['Jawaban'].str.strip() != '')
].reset_index(drop=True)

df_eval = df_eval_pool.sample(10, random_state=42).reset_index(drop=True)

eval_results = []

print(f"Menjalankan batch evaluation ({len(df_eval)} sampel)...\n")
print("=" * 70)

for idx, row in df_eval.iterrows():
    question     = row['Pertanyaan']
    ground_truth = row['Jawaban']
    result       = intent_aware_reply(question)

    eval_results.append({**result, 'ground_truth': ground_truth})

    q_short = question[:65] + '...' if len(question) > 65 else question
    r_short = result['reply'][:130] + '...' if len(result['reply']) > 130 else result['reply']
    g_short = str(ground_truth)[:100] + '...' if len(str(ground_truth)) > 100 else str(ground_truth)

    print(f"[{idx+1:02d}] Pertanyaan : {q_short}")
    print(f"      Intent    : [{result['intent_label']}] {result['category']} ({result['confidence']*100:.1f}%)")
    print(f"      Jawaban AI: {r_short}")
    print(f"      Referensi : {g_short}")
    print()

print(f"Evaluasi selesai. Total sampel: {len(eval_results)}")


Menjalankan batch evaluation (10 sampel)...

[01] Pertanyaan : izin bertanya lagi kak, kalau mengumpulkannya sebelum 20 april 20...
      Intent    : [2] Materi & Kurikulum (92.3%)
      Jawaban AI: Halo! Kamu tidak perlu khawatir, selama kamu melakukan *submit* proyek sebelum tenggat waktu (20 April 2026, pukul 23.59 WIB), mak...
      Referensi : Jika mengacu pada cohort guide, tidak dianggap telat.Perlu ditekankan bahwa 20 April 2026 hanya dead...

[02] Pertanyaan : Saya sudah membuat sebuah abstrak dan mengumpulkan nya jauh sebel...
      Intent    : [1] Capstone & Reporting (97.2%)
      Jawaban AI: Halo! Terkait status abstrak yang belum berubah, mohon lakukan langkah berikut:

1. Pastikan kembali status pengumpulan di dashboa...
      Referensi : silahkan mengubungi pihak coding camp team

[03] Pertanyaan : apakah PPT dari capstone briefing kemarin ada dibagi kak? atau te...
      Intent    : [1] Capstone & Reporting (94.8%)
      Jawaban AI: Halo! Terkait materi *capstone brief

In [24]:
# [DIAGNOSTIK] Cek model Gemini yang tersedia
# Jalankan cell ini jika terjadi error 'model not found' di cell konfigurasi.
import google.generativeai as genai

print("Model Gemini yang mendukung 'generateContent':")
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(f"  - {m.name}")


Model Gemini yang mendukung 'generateContent':
  - models/gemini-2.5-flash
  - models/gemini-2.5-pro
  - models/gemini-2.0-flash
  - models/gemini-2.0-flash-001
  - models/gemini-2.0-flash-lite-001
  - models/gemini-2.0-flash-lite
  - models/gemini-2.5-flash-preview-tts
  - models/gemini-2.5-pro-preview-tts
  - models/gemma-4-26b-a4b-it
  - models/gemma-4-31b-it
  - models/gemini-flash-latest
  - models/gemini-flash-lite-latest
  - models/gemini-pro-latest
  - models/gemini-2.5-flash-lite
  - models/gemini-2.5-flash-image
  - models/gemini-3-pro-preview
  - models/gemini-3-flash-preview
  - models/gemini-3.1-pro-preview
  - models/gemini-3.1-pro-preview-customtools
  - models/gemini-3.1-flash-lite-preview
  - models/gemini-3.1-flash-lite
  - models/gemini-3-pro-image-preview
  - models/gemini-3-pro-image
  - models/nano-banana-pro-preview
  - models/gemini-3.1-flash-image-preview
  - models/gemini-3.1-flash-image
  - models/gemini-3.5-flash
  - models/lyria-3-clip-preview
  - models/ly

## 9. Simpan Hasil Evaluasi ke JSON

In [25]:
output_path = 'genai_eval_results.json'

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(eval_results, f, ensure_ascii=False, indent=2)

print(f"Hasil evaluasi disimpan ke '{output_path}'")
print(f"Total sampel dievaluasi: {len(eval_results)}")


Hasil evaluasi disimpan ke 'genai_eval_results.json'
Total sampel dievaluasi: 10


## 10. Ringkasan & Justifikasi Penggunaan Generative AI

| Komponen | Detail |
|---|---|
| **Model Klasifikasi** | BiLSTM + Custom AttentionPooling (TF Functional API) |
| **Custom Loss** | FocalLoss |
| **Format Model** | `.keras` (TF production-ready) |
| **Generative AI** | Gemini 3.1 Flash-Lite (`models/gemini-3.1-flash-lite`) via Google Generative AI API |
| **Peran Gen AI** | Fitur *sekunder* — auto-reply berbasis intent, bukan classifier utama |
| **Input Gen AI** | Pertanyaan user + intent label dari model BiLSTM |
| **Output** | Jawaban kontekstual per kategori (Admin / Capstone / Materi / Teknis) |

> Penggunaan Gemini API hanya dipakai untuk **fitur generatif tambahan (auto-reply)**,  
> bukan sebagai classifier utama.  
> Model klasifikasi intent sepenuhnya dibangun menggunakan TensorFlow Functional API  
> dengan Custom Layer (`AttentionPooling`), Custom Loss (`FocalLoss`), dan Custom Callback.
